# 6주차. 카나메이트가 약속을 결정하다

**부제:** Supervisor/Sub-agent 최종 일정 결정: Nana/Kana delegate로 팀 회의 가능 시간 확정

## 0. 목표

이번 주에는 supervisor가 업무 tool을 직접 호출하지 않고 `nana_agent`, `kana_agent` delegate tool만 호출하는 최종 구조를 LangChain agent로 검증한다.

성취 기준:

- OpenAI API key를 사용하는 supervisor/sub-agent 흐름을 실행할 수 있다.
- Nana와 Kana의 내부 tool trace를 delegate payload에서 확인할 수 있다.
- “팀원 A/B/C와 다음 주 회의 시간을 잡아줘” 요청의 최종 후보 결정 근거를 설명할 수 있다.

![sub_agent](imgs/sub_agent.jpg)

![multi_agent](imgs/multi_agent.jpg)

## 1. 준비

6주차도 실제 OpenAI API를 호출한다. Supervisor, Nana sub-agent, Kana sub-agent가 모두 LangChain `create_agent` 기반으로 실행된다.

In [1]:
from __future__ import annotations

import json
import os
import sqlite3
import sys
import uuid
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Literal

from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain.agents.middleware import AgentMiddleware
from langchain.tools import tool
from langchain_openai import ChatOpenAI
from pydantic import BaseModel, Field


def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / ".env").exists() or (candidate / ".env.example").exists():
            return candidate
    raise RuntimeError("repo root를 찾지 못했습니다. repo 안에서 노트북을 실행하세요.")


REPO_ROOT = find_repo_root(Path.cwd())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
os.chdir(REPO_ROOT)

load_dotenv(REPO_ROOT / ".env", override=True)
OPENAI_MODEL = os.getenv("OPENAI_MODEL", "gpt-4.1-mini")
OPENAI_EMBEDDING_MODEL = os.getenv("OPENAI_EMBEDDING_MODEL", "text-embedding-3-small")

if not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError("repo 루트 .env 파일에 OPENAI_API_KEY를 설정한 뒤 다시 실행하세요.")


def make_model(max_tokens: int = 700) -> ChatOpenAI:
    return ChatOpenAI(model=OPENAI_MODEL, temperature=0, max_completion_tokens=max_tokens)


def now_iso() -> str:
    return datetime.now(timezone.utc).isoformat(timespec="seconds")


def show_json(value: Any) -> None:
    print(json.dumps(value, ensure_ascii=False, indent=2, default=str))


def final_text(agent_result: dict[str, Any]) -> str:
    return agent_result["messages"][-1].content


def extract_tool_trace(agent_result: dict[str, Any]) -> list[dict[str, Any]]:
    trace: list[dict[str, Any]] = []
    for message in agent_result.get("messages", []):
        for call in getattr(message, "tool_calls", []) or []:
            trace.append({"event": "tool_call", "tool_name": call.get("name"), "arguments": call.get("args", {})})
        if getattr(message, "type", None) == "tool":
            trace.append({"event": "tool_result", "tool_name": getattr(message, "name", None), "content": message.content})
    return trace


class DisableParallelToolCalls(AgentMiddleware):
    def wrap_model_call(self, request, handler):
        request.model_settings["parallel_tool_calls"] = False
        return handler(request)

my_schedules = [
    {"title": "개인 운동", "date": "2026-05-20", "start_time": "10:00", "end_time": "11:00"}
]
external_member_slots = [
    {"member": "A", "date": "다음 주 화요일", "start_time": "15:00"},
    {"member": "B", "date": "다음 주 화요일", "start_time": "15:00"},
    {"member": "C", "date": "다음 주 화요일", "start_time": "15:00"},
]
show_json({"model": OPENAI_MODEL, "golden_request": "팀원 A/B/C와 다음 주 회의 시간을 잡아줘"})

{
  "model": "gpt-4.1-mini",
  "golden_request": "팀원 A/B/C와 다음 주 회의 시간을 잡아줘"
}


## 2. 개념

Supervisor는 직접 개인 일정 tool이나 MCP 검색 tool을 호출하지 않는다. Nana는 내 개인 일정과 개인 RAG를 담당하고, Kana는 MCP SQLite에서 다른 사람들의 일정과 이전 대화 기록을 불러와 가능한 시간을 종합한다.

## 3. 기본 개념 실습

Nana/Kana 내부에서 사용할 업무 tool을 정의한다.

In [2]:
@tool("personal_list_schedules", description="내 개인 일정 목록을 조회한다.")
def personal_list_schedules(date_range: str = "next_week") -> str:
    """List my personal schedules."""
    return json.dumps({"date_range": date_range, "schedules": my_schedules}, ensure_ascii=False)


@tool("search_previous_conversations", description="MCP SQLite에서 팀원들의 이전 일정 대화를 검색한다.")
def search_previous_conversations(members: list[str]) -> str:
    """Search external member history through MCP SQLite."""
    return json.dumps({"members": members, "conversation_ids": [f"conv-{member.lower()}" for member in members]}, ensure_ascii=False)


@tool("extract_schedules_from_history", description="이전 대화에서 팀원별 가능 시간을 추출한다.")
def extract_schedules_from_history(conversation_ids: list[str]) -> str:
    """Extract member schedule candidates from history."""
    return json.dumps({"schedules": external_member_slots, "conversation_ids": conversation_ids}, ensure_ascii=False)


@tool("decide_final_slot", description="팀원 가능 시간 후보만 비교해 최종 회의 시간 후보를 결정한다. 개인 일정 충돌은 판단하지 않는다.")
def decide_final_slot(candidate_slots: list[str]) -> str:
    """Decide the final meeting slot."""
    return json.dumps({"final_slot": "다음 주 화요일 15:00", "external_reason": "A/B/C 가능 시간이 모두 겹치는 후보", "candidates": candidate_slots}, ensure_ascii=False)

show_json({"nana_tools": ["personal_list_schedules"], "kana_tools": ["search_previous_conversations", "extract_schedules_from_history", "decide_final_slot"]})

{
  "nana_tools": [
    "personal_list_schedules"
  ],
  "kana_tools": [
    "search_previous_conversations",
    "extract_schedules_from_history",
    "decide_final_slot"
  ]
}


## 4. 카나메이트 확장 예제

Delegate tool은 sub-agent를 실행하고, sub-agent 내부 trace를 JSON payload로 supervisor에게 돌려준다.

In [3]:
@tool("nana_agent", description="내 개인 일정 생성/조회/삭제와 개인 RAG 검색을 Nana sub-agent에게 위임한다.")
def nana_agent(request: str) -> str:
    """Delegate personal schedule work to Nana."""
    agent = create_agent(
        model=make_model(700),
        tools=[personal_list_schedules],
        system_prompt=("너는 개인 메이트 Nana다. 어떤 회의 요청이든 반드시 personal_list_schedules tool을 먼저 호출해 " "내 개인 일정 충돌 여부를 확인한 뒤 답한다. tool 호출 없이 답하지 않는다."),
    )
    result = agent.invoke({"messages": [{"role": "user", "content": request}]})
    return json.dumps({"agent": "nana", "answer": final_text(result), "trace": extract_tool_trace(result)}, ensure_ascii=False)


@tool("kana_agent", description="MCP SQLite에서 다른 사람들의 일정과 이전 대화 기록을 불러와 최종 회의 시간을 결정한다.")
def kana_agent(request: str) -> str:
    """Delegate group scheduling work to Kana."""
    agent = create_agent(
        model=make_model(900),
        tools=[search_previous_conversations, extract_schedules_from_history, decide_final_slot],
        system_prompt=(
            "너는 그룹 메이트 Kana다. 팀원 A/B/C 회의 요청을 받으면 반드시 "
            "search_previous_conversations, extract_schedules_from_history, decide_final_slot을 이 순서로 모두 호출해 팀원 가능 시간 후보만 결정한다. "
            "내 개인 일정 충돌 여부는 판단하지 말고, 개인 일정과 충돌하지 않는다고 말하지 않는다. tool 호출 없이 답하지 않는다."
        ),
    )
    result = agent.invoke({"messages": [{"role": "user", "content": request}]})
    return json.dumps({"agent": "kana", "answer": final_text(result), "trace": extract_tool_trace(result)}, ensure_ascii=False)

## 5. 확장 예제 실행

Supervisor에게는 `nana_agent`, `kana_agent` delegate tool만 제공한다.

In [4]:
supervisor = create_agent(
    model=make_model(1000),
    tools=[nana_agent, kana_agent],
    middleware=[DisableParallelToolCalls()],
    system_prompt=(
        "너는 카나메이트 supervisor다. 업무 tool을 직접 처리하지 말고 delegate tool만 호출한다. "
        "모든 회의 시간 결정 요청에서 반드시 먼저 nana_agent를 호출해 내 개인 일정 충돌을 확인하고, "
        "그 다음 반드시 kana_agent를 호출해 팀원 A/B/C의 이전 대화와 최종 일정 결정을 확인한다. "
        "두 tool_result를 모두 받은 뒤에만 최종 답변을 작성한다."
    ),
)

final_request = "팀원 A/B/C와 다음 주 회의 시간을 잡아줘. 내 개인 일정도 확인하고, 팀원들의 이전 대화 기록도 확인해서 최종 시간을 정해줘."
supervisor_result = supervisor.invoke({"messages": [{"role": "user", "content": final_request}]})
supervisor_trace = extract_tool_trace(supervisor_result)

print(final_text(supervisor_result))
show_json(supervisor_trace)

delegate_payloads = []
for event in supervisor_trace:
    if event["event"] == "tool_result" and event["tool_name"] in {"nana_agent", "kana_agent"}:
        delegate_payloads.append(json.loads(event["content"]))

called_delegate_tools = [event["tool_name"] for event in supervisor_trace if event["event"] == "tool_call"]
assert "nana_agent" in called_delegate_tools
assert "kana_agent" in called_delegate_tools
delegate_event_order = [(event["event"], event["tool_name"]) for event in supervisor_trace]
assert delegate_event_order.index(("tool_call", "nana_agent")) < delegate_event_order.index(("tool_result", "nana_agent"))
assert delegate_event_order.index(("tool_result", "nana_agent")) < delegate_event_order.index(("tool_call", "kana_agent"))
assert delegate_event_order.index(("tool_call", "kana_agent")) < delegate_event_order.index(("tool_result", "kana_agent"))
kana_payload = next(payload for payload in delegate_payloads if payload["agent"] == "kana")
assert "개인 일정과도 충돌하지" not in kana_payload["answer"]
assert "external_reason" in json.dumps(kana_payload["trace"], ensure_ascii=False)
show_json({"called_delegate_tools": called_delegate_tools, "delegate_payloads": delegate_payloads})

내 개인 일정에 따르면, 다음 주 화요일 10:00 - 11:00에 개인 운동이 예정되어 있습니다. 그러나 팀원 A, B, C의 이전 대화 기록에 따르면, 모두 다음 주 화요일 15:00에 회의가 가능하다고 합니다.

따라서, 최종 회의 시간은 **다음 주 화요일 15:00**로 결정되었습니다.
[
  {
    "event": "tool_call",
    "tool_name": "nana_agent",
    "arguments": {
      "request": "내 개인 일정 확인"
    }
  },
  {
    "event": "tool_result",
    "tool_name": "nana_agent",
    "content": "{\"agent\": \"nana\", \"answer\": \"다음 주 개인 일정은 다음과 같습니다:\\n\\n- **개인 운동**: 2026-05-20, 10:00 - 11:00\\n\\n이 일정을 참고하여 다른 회의 요청을 해주시면 됩니다.\", \"trace\": [{\"event\": \"tool_call\", \"tool_name\": \"personal_list_schedules\", \"arguments\": {}}, {\"event\": \"tool_result\", \"tool_name\": \"personal_list_schedules\", \"content\": \"{\\\"date_range\\\": \\\"next_week\\\", \\\"schedules\\\": [{\\\"title\\\": \\\"개인 운동\\\", \\\"date\\\": \\\"2026-05-20\\\", \\\"start_time\\\": \\\"10:00\\\", \\\"end_time\\\": \\\"11:00\\\"}]}\"}]}"
  },
  {
    "event": "tool_call",
    "tool_name": "kana_agent",
    "arguments": {
      "request

## 6. 회고

확인 질문:

1. Supervisor가 직접 `personal_list_schedules`나 MCP 검색 tool을 호출하지 않는 이유는 무엇인가?
2. Nana와 Kana의 책임은 어떤 기준으로 나뉘는가?
3. 최종 일정 결정에서 가능한 시간 후보와 선택 이유를 함께 보여줘야 하는 이유는 무엇인가?

작은 응용 과제: 팀원 한 명이 선택된 시간에 불가능하다고 가정하고 `decide_final_slot` 결과와 supervisor 답변이 어떻게 달라져야 하는지 작성한다.